# Modeling — Allstate Claims Severity

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.load_data import load_train
from src.features import fit_preprocessor, transform_features, split_features_target

df = load_train('../data/raw/train.csv')

train: (188319, 132)


  dropped 1 row(s) with null loss


## Split and preprocess

In [2]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

preprocessor = fit_preprocessor(train_df)

X_train, y_train = split_features_target(train_df)
X_val, y_val = split_features_target(val_df)

X_train = transform_features(X_train, preprocessor)
X_val = transform_features(X_val, preprocessor)

y_train_log = np.log1p(y_train)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (150654, 130), Val: (37664, 130)


## Results

| Model | MAE | RMSE |
|-------|-----|------|
| HistGradientBoosting | 1151.53 | 1934.26 |
| LightGBM | 1151.92 | 1934.37 |
| XGBoost | 1155.49 | 1936.01 |
| Median baseline | 1796.96 | 2998.64 |

**Winner: HistGradientBoosting** (marginally ahead of LightGBM)

All three boosted tree models beat the median baseline by ~$645 MAE — a 36% improvement. The three models are very close to each other; LightGBM and HistGradientBoosting are essentially tied.

In [3]:
def evaluate(name, preds_log):
    preds = np.expm1(preds_log)
    return {
        "model": name,
        "MAE": round(mean_absolute_error(y_val, preds), 2),
        "RMSE": round(root_mean_squared_error(y_val, preds), 2),
    }

results = []

# Median baseline
median_pred = np.full(len(y_val), np.log1p(y_train.median()))
results.append(evaluate("Median baseline", median_pred))

# HistGradientBoostingRegressor
hgb = HistGradientBoostingRegressor(random_state=42)
hgb.fit(X_train, y_train_log)
results.append(evaluate("HistGradientBoosting", hgb.predict(X_val)))

# LightGBM
lgbm = LGBMRegressor(random_state=42, verbose=-1)
lgbm.fit(X_train, y_train_log)
results.append(evaluate("LightGBM", lgbm.predict(X_val)))

# XGBoost
xgb = XGBRegressor(random_state=42, verbosity=0)
xgb.fit(X_train, y_train_log)
results.append(evaluate("XGBoost", xgb.predict(X_val)))

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_df

,model,MAE,RMSE
0,HistGradientBoosting,1151.53,1934.26
1,LightGBM,1151.92,1934.37
2,XGBoost,1155.49,1936.01
3,Median baseline,1796.96,2998.64
